In [1]:
import pandas as pd
import os

In [2]:
# goal: make an outer join mechanism table (mirroring vsm table) (all mechanism data possible to linker table)

# need:
## linker table
## mechanism tables

In [3]:
def load_linker(chrom):
    df_l_c =  pd.read_csv(
            f'gs://genetics-gym/linkers/linker_missense_enst_transcript_aa_uniprot_by_chrom_tsv/linker_missense_enst_transcript_aa_uniprot_{chrom}.tsv.bgz',
            sep='\t',
            compression='gzip'
        )
    
    df_l_c = df_l_c[
        ['chrom', 'pos', 'ref', 'alt', 'ensg', 'enst',
         'uniprot_isoform', 'uniprot_id', 'aa_pos', 'aa_ref', 'aa_alt']
    ]

    return df_l_c

In [4]:
g2p_f_categorical = {'Secondary structure (DSSP 3-state)*': 
                     ['C (loop/coil)', 'H (helix)', 'B (undefined)']}
g2p_quantitative = {'fpocket: druggability score*': 'druggability_score'}
g2p_f_binary = {'Compositional bias (UniProt)': 'compositional_bias', # would expect these variants to be benign
                'Cross-link (UniProt)': 'cross_link',
                'Disulfide bond (UniProt)': 'disulfide_bond', 
                'Domain (UniProt)': 'domain',
                'Glycosylation (UniProt)': 'glycosylation', 
                'Lipidation (UniProt)': 'lipidation', 
                'Motif (UniProt)': 'motif', 
                'Region (UniProt)': 'region',
                'Repeat (UniProt)': 'repeat',
                'Transmembrane (UniProt)': 'transmembrane',
                'Zinc finger (UniProt)': 'zinc_finger',
                'Acetylation': 'acetylation',
                'Methylation': 'methylation', 
                'Phosphorylation': 'phosphorylation', 
                'SUMOylation': 'SUMOylation', 
                'Ubiquitination': 'ubiquitination',
                'Intra-chain Hydrogen bond (AlphaFold2)': 'intra_chain_hydrogen_bond',
                'Intra-chain Non-bonded interaction (AlphaFold2)': 'intra_chain_nonbonded_interaction',
                'Intra-chain Salt bridge (AlphaFold2)': 'intra_chain_salt_bridge',
                'Signal (UniProt)': 'signal', 
                'Initiator methionine (UniProt)': 'initiator_methionine', 
                'Propeptide (UniProt)': 'propeptide', 
                'Peptide (UniProt)': 'peptide', 
                'Chain (UniProt)': 'chain', 
                'Topological domain (UniProt)': 'topological_domain', 
                'Intramembrane (UniProt)': 'intramembrane', 
                'Transit peptide (UniProt)': 'transit_peptide'}
g2p_change = {'Molar mass (g/mol)': 'delta_molar_mass', 
              'Hydropathy': 'delta_hydropathy'}
g2p_sites = ['Active site (UniProt)', 'Binding site (UniProt)', 'Site (UniProt)']

In [5]:
# feature data
# isoform information
df_p = pd.read_csv('../reference_data/protein_sequence_guide.tsv', sep='\t')
df_p = df_p[df_p.uniprot_canonical]
df_p = df_p[['uniprot_id', 'uniprot_isoform']].drop_duplicates()
# pLDDT
df_plddt = pd.read_csv('../reference_data/plddt_missense_variants_gr38.tsv.gz', sep='\t', compression='gzip')
# add in isoform information
df_plddt = df_plddt.merge(df_p, how='inner', on='uniprot_id')
df_plddt = df_plddt[['aa_pos', 'aa_ref', 'avg_plddt', 'uniprot_isoform']]
df_plddt = df_plddt.rename(columns={'avg_plddt': 'plddt'})
# mutation rate
df_mu = pd.read_csv('../reference_data/mutation_rate.tsv.bgz', sep='\t', compression='gzip')
# foldX
df_foldx = pd.read_csv('../reference_data/foldx/foldx_max_plddt_clean.tsv.gz', sep='\t', compression='gzip')
# add in isoform information - skipping for now - how comfortable are we with dropping data? or having questionable data?
df_foldx = df_foldx.merge(df_p, how='inner', on='uniprot_id')
df_foldx = df_foldx[['aa_pos', 'aa_ref', 'aa_alt', 'foldx_ddg', 'uniprot_isoform']]
# rSASA
df_rsasa = pd.read_csv('../reference_data/rsasa_missense_variants_gr38.tsv.gz', sep='\t', compression='gzip')
# add in isoform information
df_rsasa = df_rsasa.merge(df_p, how='inner', on='uniprot_id')
df_rsasa = df_rsasa[['aa_pos', 'aa_ref', 'rsasa', 'uniprot_isoform']]
# af2bind
af2bind = pd.read_csv('../reference_data/af2bind/expanded_pockets.tsv', sep='\t')
# get individual residues in pockets
af2bind=af2bind[['uniprot', 'expanded_pocket']]
def parse_int_list(s):
    # remove brackets and newlines, then split on whitespace
    return list(map(int, s.strip('[]').replace('\n', ' ').split()))
af2bind['expanded_pocket'] = af2bind['expanded_pocket'].apply(parse_int_list)
af2bind_vars = af2bind.explode('expanded_pocket')
af2bind_vars = af2bind_vars.drop_duplicates()
af2bind_vars = af2bind_vars.rename(columns={'uniprot': 'uniprot_id', 'expanded_pocket': 'aa_pos'})
af2bind_vars['af2bind_pocket'] = True
# af2bind only gives ID not isoform - but should be based off of the canonical isoform?
af2bind_vars = af2bind_vars.merge(df_p, how='inner', on=['uniprot_id']) # gives us the canonical isoform (to the best of my ability)
af2bind_vars = af2bind_vars[["aa_pos", "af2bind_pocket", "uniprot_isoform"]]
# # AA properties
non_polar = ['A', 'G', 'I', 'L', 'M', 'P', 'V']
polar = ['C', 'N', 'Q', 'S', 'T']
charged = ['D', 'E', 'H', 'K', 'R']
aromatic = ['F', 'W', 'Y']
# aa_property = {'A': 'Nonpolar/Neutral', 'C': 'Polar/Neutral', 'D': 'Charged', 'E': 'Charged', 'F': 'Aromatic', 'G': 'Nonpolar/Neutral', 
#            'H': 'Charged', 'I': 'Nonpolar/Neutral', 'K': 'Charged', 'L': 'Nonpolar/Neutral', 'M': 'Nonpolar/Neutral', 'N': 'Polar/Neutral', 
#            'P': 'Nonpolar/Neutral', 'Q': 'Polar/Neutral', 'R': 'Charged', 'S': 'Polar/Neutral', 'T': 'Polar/Neutral', 
#            'V': 'Nonpolar/Neutral', 'W': 'Aromatic', 'Y': 'Aromatic'}
# # alt disorder metric - mobidb
df_mobidb = pd.read_csv('../reference_data/mobidb/mobidb_disorder_per_residue.tsv.gz', sep='\t', compression='gzip')
# identity information of hydropathy and molar mass change per amino acid ref-alt pair
h_mm_df = pd.read_csv('../reference_data/hydrop_mm_guide.tsv', sep='\t')
#h_mm_df = h_mm_df[['aa_ref', 'aa_alt', 'delta_hydropathy', 'delta_molar_mass']]
# blosum scores
blosum_df = pd.read_csv('../reference_data/blosum/blosum_all.tsv', sep='\t')

In [2]:
chromosomes = [str(i) for i in range(1,23)] + ['X']
linker_key_cols = ['chrom', 'pos', 'ref', 'alt', 'ensg']

In [7]:
# g2p features
df_g2p = pd.read_csv('../reference_data/g2p_data_clean.tsv.gz', sep='\t', compression='gzip')

/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/2529505913.py:2: DtypeWarning: Columns (5,6,8,9,10,12,13,14,18,23,24,25,26,28,29,30,33,34,35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  df_g2p = pd.read_csv('../analysis_vsm_mech/g2p_data/g2p_data_clean.tsv.gz', sep='\t', compression='gzip')


In [8]:
# results directory
os.makedirs('mech_file_by_chrom', exist_ok=True)

In [12]:
for chrom in chromosomes:
    print(f'Chromosome {chrom}')

    # linker file for chromosome
    df_l_c = load_linker(chrom)

    # add each feature
    f_list = []

    print(len(df_l_c))
    print(len(df_l_c.ensg.unique()))

    print('starting non-g2p features')
    # foldx
    f = 'foldx_ddg'
    f_list = f_list+[f]
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref', 'aa_alt']
    # list of valid uniprot ids
    uni_iso = df_foldx.uniprot_isoform.unique()
    # add to linker table
    df_c = df_l_c.merge(df_foldx, how='left', on=join_list)
    # fill in NA with 0 where we have foldx data for the isoform (variants with low plddt) 
    has_foldx = df_c["uniprot_isoform"].isin(uni_iso)
    df_c.loc[
        has_foldx & df_c[f].isna(),
        f
    ] = 0
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)
    
    # mutation rate
    f = 'mu_snp'
    f_list = f_list+[f]
    join_list = ['chrom', 'pos', 'ref', 'alt']
    df_c = df_c.merge(df_mu, how='left', on=join_list)
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)
    
    # rsasa
    f = 'rsasa'
    f_list = f_list+[f]
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    df_c = df_c.merge(df_rsasa, how='left', on=join_list)
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)
    
    # plddt
    f = 'plddt'
    f_list = f_list+[f]
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    df_c = df_c.merge(df_plddt, how='left', on=join_list)
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)

    # mobidb
    f = 'mobidb_score'
    f_list = f_list+[f]
    join_list = ['uniprot_isoform', 'aa_pos']
    df_c = df_c.merge(df_mobidb, how='left', on=join_list)
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)
    
    # af2bind
    f = 'af2bind_pocket'
    f_list = f_list+[f]
    # list of valid uniprot ids
    uni_iso = af2bind_vars.uniprot_isoform.unique()
    # add to linker table
    df_c = df_c.merge(af2bind_vars, how='left', on=['uniprot_isoform', 'aa_pos'])
    # fill in NA with False where we have af2bind data for the isoform (variants not on pocket) 
    has_af2bind = df_c["uniprot_isoform"].isin(uni_iso)
    df_c.loc[
        has_af2bind & df_c[f].isna(),
        f
    ] = False
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)

    # amino acid properties
    f_list = f_list+['nonpolar_neutral', 'polar_neutral', 'charged', 'aromatic']
    df_c['nonpolar_neutral'] = df_c.aa_ref.isin(non_polar)
    df_c['polar_neutral'] = df_c.aa_ref.isin(polar)
    df_c['charged'] = df_c.aa_ref.isin(charged)
    df_c['aromatic'] = df_c.aa_ref.isin(aromatic)

    print('starting g2p features')
    # g2p_change
    join_list = ['aa_ref', 'aa_alt']
    f_list = f_list + ['delta_hydropathy', 'delta_molar_mass', 'hydropathy', 'molar_mass']
    df_c = df_c.merge(h_mm_df, how='left', on=join_list)
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)

    # G2P features - Quantitative - just druggability score now
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    for f in g2p_quantitative.keys():
        df_g2p_f = df_g2p[join_list+[f]]
        df_g2p_f = df_g2p_f[df_g2p_f[f].notna()].reset_index(drop=True)
        df_g2p_f = df_g2p_f.rename(columns={f: g2p_quantitative[f]})
        if g2p_quantitative[f] == 'druggability_score': # this feature needs specific cleaning
            df_g2p_f[g2p_quantitative[f]] = df_g2p_f[g2p_quantitative[f]].astype(str)
            df_g2p_f[g2p_quantitative[f]] = df_g2p_f[g2p_quantitative[f]].str.split(': ').str[-1]
            df_g2p_f[g2p_quantitative[f]] = df_g2p_f[g2p_quantitative[f]].astype(float)
        
        f_list = f_list+[g2p_quantitative[f]]
        df_c = df_c.merge(df_g2p_f, how='left', on=join_list)
        df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)
    
    # G2P features - binary
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    for f in g2p_f_binary.keys():
        df_g2p_f = df_g2p[join_list+[f]]
        df_g2p_f[f] = df_g2p_f[f].notna()
        df_g2p_f = df_g2p_f.rename(columns={f: g2p_f_binary[f]})
        # list of valid uniprot ids
        uni_iso = df_g2p_f.uniprot_isoform.unique()
        f_list = f_list+[g2p_f_binary[f]]
        df_c = df_c.merge(df_g2p_f, how='left', on=join_list)
        # fill in NA with False where we have g2p data for the isoform (variants do not have annotation) 
        has_g2p = df_c["uniprot_isoform"].isin(uni_iso)
        df_c.loc[
            has_g2p & df_c[g2p_f_binary[f]].isna(),
            g2p_f_binary[f]
        ] = False
        df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)
    
    # G2P features - categorical - just secondary structure
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    fc = 'Secondary structure (DSSP 3-state)*'
    df_g2p_f = df_g2p[join_list+[fc]]
    ss_3 = {'C (loop/coil)': 'ss_loop_coil', 'H (helix)': 'ss_helix', 'B (undefined)': 'ss_undefined'}
    for ss in ss_3:
        df_g2p_f[ss_3[ss]] = df_g2p_f[fc]==ss
        f_list = f_list+[ss_3[ss]]
    # list of valid uniprot ids
    uni_iso = df_g2p_f.uniprot_isoform.unique()
    df_c = df_c.merge(df_g2p_f, how='left', on=join_list)
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)

    # g2p_sites - combine binding sites, active sites, and other sites
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    df_g2p_f = df_g2p[join_list+g2p_sites]
    df_g2p_f['all_uniprot_sites'] = ((df_g2p_f[g2p_sites[0]].notna()) | (df_g2p_f[g2p_sites[1]].notna()) | (df_g2p_f[g2p_sites[2]].notna()))
    df_g2p_f['all_uniprot_sites'] = df_g2p_f['all_uniprot_sites'].fillna(False)
    # list of valid uniprot ids
    uni_iso = df_g2p_f.uniprot_isoform.unique()
    # now merge with g2p data
    f_list = f_list+['all_uniprot_sites']
    df_c = df_c.merge(df_g2p_f, how='left', on=join_list)
    # fill in NA with False where we have g2p data for the isoform (variants do not have annotation) 
    has_g2p = df_c['uniprot_isoform'].isin(uni_iso)
    df_c.loc[
        has_g2p & df_c['all_uniprot_sites'].isna(),
        'all_uniprot_sites'
    ] = False
    df_c = df_c.drop_duplicates(subset=linker_key_cols+f_list)

    # blosum
    print('starting blosum scores')
    df_c = df_c.merge(blosum_df, on=['aa_ref', 'aa_alt'])
    f_list = f_list + ['blosum62', 'blosum45', 'blosum80']

    # save chrom file
    df_c = df_c[linker_key_cols + f_list]
    df_c.to_csv(f'mech_file_by_chrom/linker_mech_chrom_{chrom}.tsv.gz', sep='\t', compression='gzip', index=False)

Chromosome 1


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/3336297406.py:2: DtypeWarning: Columns (11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_l_c =  pd.read_csv(


25789010
2057
starting non-g2p features
starting g2p features


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/324950464.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_g2p_f[f] = df_g2p_f[f].notna()
/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/324950464.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_g2p_f[f] = df_g2p_f[f].notna()
/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/324950464.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .

starting blosum scores
Chromosome 2


...

starting blosum scores
Chromosome X


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/3336297406.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_l_c =  pd.read_csv(


10283099
857
starting non-g2p features
starting g2p features


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/324950464.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_g2p_f[f] = df_g2p_f[f].notna()
/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/324950464.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_g2p_f[f] = df_g2p_f[f].notna()
/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_23897/324950464.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .

starting blosum scores


In [ ]:
# combine chroms into one (parquet)
df = pd.DataFrame()
for c in chromosomes:
    df_c = pd.read_csv(f'mech_file_by_chrom/linker_mech_chrom_{c}.tsv.gz', sep='\t', compression='gzip')
    df = pd.concat([df, df_c], ignore_index=True)
df["foldx_ddg"] = pd.to_numeric(df["foldx_ddg"], errors="coerce")
df.to_parquet("mechanisms_all.parquet", index=False)

/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_37973/3919462204.py:4: DtypeWarning: Columns (10,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50) have mixed types. Specify dtype option on import or set low_memory=False.
  df_c = pd.read_csv(f'mech_file_by_chrom/linker_mech_chrom_{c}.tsv.gz', sep='\t', compression='gzip')
/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_37973/3919462204.py:4: DtypeWarning: Columns (10,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50) have mixed types. Specify dtype option on import or set low_memory=False.
  df_c = pd.read_csv(f'mech_file_by_chrom/linker_mech_chrom_{c}.tsv.gz', sep='\t', compression='gzip')
/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_37973/3919462204.py:4: DtypeWarning: Columns (10,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50) have mixed types. Specify dtype option o

In [ ]:
df.head()

In [ ]:
df.columns